# Instance page: demo on the running example

Page doc: `docs/pages/instances.md`; user-facing description: `overview/module/Instance Page.md` (outside this repo).
Code: package `ocelescope_module_location_based_ocpm.instances`; this notebook only imports and calls its functions.
All values use the low-level location $loc = loc^k$ of the selected level (no visits).

In [ ]:
import json
from pathlib import Path

import plotly.graph_objects as go
import polars as pl
from ocelescope import OCEL

from ocelescope_module_location_based_ocpm.derivation.config import DerivationConfig
from ocelescope_module_location_based_ocpm.derivation.pipeline import derive
from ocelescope_module_location_based_ocpm.instances.charts import processing_events, selection_lineage, timeline_figure
from ocelescope_module_location_based_ocpm.instances.profile import (
    co_handled,
    hu_profile,
    lifecycle_basics,
    location_profile,
    ordered_traces,
    query_profile,
    time_per_location_kind,
    transformation_profile,
)

EXAMPLE = Path("../data/running_example")
pl.Config.set_tbl_cols(40)
pl.Config.set_tbl_rows(40)

## 1. Inputs

The running example with its full configuration (`configuration.json`): Level 3, high-level locations, vehicle T and storage unit B1 from `inputs/locations.csv`, processing operations = transformation events and *label*.
`derive` gives the relations (with $loc^k$ and $agg(loc^k)$), the transformation roles, the handling units and the location context.

In [ ]:
config = DerivationConfig.model_validate(json.loads((EXAMPLE / "configuration.json").read_text()))
tables = {path.name: pl.read_csv(path, infer_schema=False) for path in (EXAMPLE / "inputs").glob("*.csv")}
derivation = derive(OCEL.read(EXAMPLE / "running_example.sqlite"), config, tables)
processing = processing_events(derivation)
derivation.roles, derivation.context, processing

## 2. HU profile, step by step

`ordered_traces` puts the relations of every handling unit in `seq` order (time, AS4, event id) and numbers them (`pos`); `loc` and `hl` are the columns of the selected level.

In [ ]:
traces = ordered_traces(derivation.relations, derivation.loc, derivation.hl, processing)
traces.filter(pl.col("hu") == "c1").collect()

`lifecycle_basics`: object type, events, first and last timestamp, lifespan (seconds), movements, processing operations and their activities.

In [ ]:
lifecycle_basics(traces).sort("hu").collect()

`location_profile`: distinct low- and high-level locations, first and last location, location changes (ignoring ⊥ in between), locatedness.

In [ ]:
location_profile(traces).sort("hu").collect()

`time_per_location_kind`: after relation $r_j$ the handling unit is at $loc(r_j)$ until $r_{j+1}$; this time goes to *vehicle* ($V$), *storage* ($W$), *other* or *unlocated* (⊥). The four times add up to the lifespan (c1: 2 h, 1 h, 8 h, 0 = 11 h).

In [ ]:
time_per_location_kind(traces, derivation.context).sort("hu").collect()

`transformation_profile`: creation event, final consumption (a consumption at the last timestamp), partial consumptions (the other consumptions), parents, children, siblings and the lifecycle class (`full`, `produced`, `consumed`, `open`).

In [ ]:
transformation_profile(derivation.roles, traces).sort("hu").collect()

`co_handled`: number of other handling units sharing at least one event.

In [ ]:
co_handled(traces).sort("hu").collect()

`hu_profile` runs all steps and joins them (also handling units without events); `query_profile` searches (all words), filters by object type, sorts (nulls last) and pages, as the table on the page does.

In [ ]:
profile = hu_profile(derivation.relations, derivation.handling_units, derivation.roles, derivation.context, derivation.loc, derivation.hl, processing)
page, total = query_profile(profile, search="", sort_by="time_storage", descending=True, limit=3)
total, page.select("hu", "lifecycle", "time_vehicle", "time_storage", "time_other", "time_unlocated")

## 3. Lineage of a selection

`selection_lineage` merges the lineage graphs of several handling units; selected ones are `focal`. Here c2 and c3 (c3 has no lineage), both directions, two generations, then c2 with events and siblings.

In [ ]:
lineage = selection_lineage(derivation, ["c2", "c3"], "both", 2)
lineage.nodes, lineage.edges

In [ ]:
lineage = selection_lineage(derivation, ["c2"], "both", 2, with_events=True, with_siblings=True)
lineage.nodes, lineage.edges

## 4. HU timeline

`timeline_figure` draws the HU timeline (resource `docs/resources/object-timeline.md`) of handling units at the selected level as Plotly JSON, as the page receives it; here the whole lineage of c2 with its sibling, ordered by first event.

In [ ]:
figure, notes = timeline_figure(derivation, ["c2", "s1", "c1", "c4"], by_first_event=True)
print(notes)
go.Figure(figure)